[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/06_Functions/Functions_Apply.ipynb)

# 1.6 ONNX Functions — Hands-On Practice

Create, use, and compose custom ONNX functions.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | [Setup](#section-1) | Imports |
| 2 | [Exercise 1: AbsLinearRegression](#section-2) | Create and use a function |
| 3 | [Exercise 2: Chain Functions](#section-3) | Use same function twice |
| 4 | [Exercise 3: ReLU Linear Function](#section-4) | Y = Relu(XA + B) |
| 5 | [Exercise 4: Multiple Functions](#section-5) | Define and use two functions |
| 6 | [Exercise 5: Inspect Functions](#section-6) | Programmatic function analysis |
| 7 | [Visualization: Function Library](#section-7) | Graph of all defined functions |
| 8 | [Challenge: Sigmoid Linear Unit](#section-8) | SiLU(x) = x · sigmoid(x) |

<a id='section-1'></a>
## Section 1: Setup

In [ ]:
# !pip install onnx onnxruntime matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt

from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid, make_function)
from onnx.checker import check_model
import onnxruntime as ort

DOMAIN = 'my_ops'
OPSETS = [make_opsetid('', 14), make_opsetid(DOMAIN, 1)]

print('Setup complete!')

<a id='section-2'></a>
## Section 2: Exercise 1 — AbsLinearRegression Function

### Task

Define $\text{AbsLR}(X, A, B) = |XA + B|$ as a reusable function and use it in a model.

In [ ]:
abs_lr = make_function(
    DOMAIN, 'AbsLinearRegression',
    ['X', 'A', 'B'], ['Y'],
    [make_node('MatMul', ['X', 'A'], ['XA']),
     make_node('Add', ['XA', 'B'], ['XAB']),
     make_node('Abs', ['XAB'], ['Y'])],
    OPSETS, [])

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

graph = make_graph(
    [make_node('AbsLinearRegression', ['X', 'A', 'B'], ['Y'], domain=DOMAIN)],
    'ex1', [X, A, B], [Y])
model = make_model(graph, opset_imports=OPSETS, functions=[abs_lr])
check_model(model)

sess = ort.InferenceSession(
    model.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.array([[1, 2], [3, 4]], dtype=np.float32)
a = np.array([[0.5], [-0.3]], dtype=np.float32)
b = np.array([[-5.0]], dtype=np.float32)

result = sess.run(None, {'X': x, 'A': a, 'B': b})[0]
expected = np.abs(x @ a + b)

print(f'AbsLinearRegression(X, A, B) = |XA + B|')
print(f'  Result:   {result.flatten()}')
print(f'  Expected: {expected.flatten()}')
print(f'  Match: {np.allclose(result, expected)}')

<a id='section-3'></a>
## Section 3: Exercise 2 — Chain Functions

### Task

Use `AbsLinearRegression` **twice** in a chain: $\text{mid} = \text{AbsLR}(X, A, B)$, then $Y = \text{AbsLR}(\text{mid}, A, B)$.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 2])
A = make_tensor_value_info('A', TensorProto.FLOAT, [2, 2])
B = make_tensor_value_info('B', TensorProto.FLOAT, [1, 2])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 2])

graph = make_graph(
    [make_node('AbsLinearRegression', ['X', 'A', 'B'], ['mid'], domain=DOMAIN),
     make_node('AbsLinearRegression', ['mid', 'A', 'B'], ['Y'], domain=DOMAIN)],
    'chain', [X, A, B], [Y])
model_chain = make_model(graph, opset_imports=OPSETS, functions=[abs_lr])
check_model(model_chain)

sess_c = ort.InferenceSession(
    model_chain.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.array([[1, -1]], dtype=np.float32)
a = np.eye(2, dtype=np.float32)
b = np.array([[0.5, -0.5]], dtype=np.float32)

result = sess_c.run(None, {'X': x, 'A': a, 'B': b})[0]

mid_np = np.abs(x @ a + b)
y_np = np.abs(mid_np @ a + b)

print(f'Chained function:')
print(f'  X = {x}')
print(f'  After 1st: |XA+B| = {mid_np}')
print(f'  After 2nd: ||XA+B|A+B| = {y_np}')
print(f'  ONNX: {result}')
print(f'  Match: {np.allclose(result, y_np)}')

<a id='section-4'></a>
## Section 4: Exercise 3 — ReLU Linear Function

### Task

Define $\text{ReluLinear}(X, W, b) = \text{Relu}(XW + b)$ — a standard neural network layer.

In [ ]:
relu_linear = make_function(
    DOMAIN, 'ReluLinear',
    ['X', 'W', 'b'], ['Y'],
    [make_node('MatMul', ['X', 'W'], ['XW']),
     make_node('Add', ['XW', 'b'], ['pre']),
     make_node('Relu', ['pre'], ['Y'])],
    OPSETS, [])

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
W = make_tensor_value_info('W', TensorProto.FLOAT, [4, 3])
b = make_tensor_value_info('b', TensorProto.FLOAT, [3])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 3])

graph = make_graph(
    [make_node('ReluLinear', ['X', 'W', 'b'], ['Y'], domain=DOMAIN)],
    'relu_layer', [X, W, b], [Y])
model_relu = make_model(graph, opset_imports=OPSETS, functions=[relu_linear])
check_model(model_relu)

sess_r = ort.InferenceSession(
    model_relu.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.random.randn(5, 4).astype(np.float32)
w = np.random.randn(4, 3).astype(np.float32) * 0.1
b_val = np.zeros(3, dtype=np.float32)

result = sess_r.run(None, {'X': x, 'W': w, 'b': b_val})[0]
expected = np.maximum(0, x @ w + b_val)

print(f'ReluLinear: Y = Relu(XW + b)')
print(f'  Shape: {result.shape}')
print(f'  Match: {np.allclose(result, expected, atol=1e-6)}')
print(f'  All non-negative: {(result >= 0).all()}')

<a id='section-5'></a>
## Section 5: Exercise 4 — Multiple Functions in One Model

### Task

Define **two** functions and use both in the same model: `ReluLinear` for hidden layers and `AbsLinearRegression` for the output.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
W1 = make_tensor_value_info('W1', TensorProto.FLOAT, [4, 8])
b1 = make_tensor_value_info('b1', TensorProto.FLOAT, [8])
W2 = make_tensor_value_info('W2', TensorProto.FLOAT, [8, 2])
b2 = make_tensor_value_info('b2', TensorProto.FLOAT, [1, 2])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 2])

graph = make_graph(
    [make_node('ReluLinear', ['X', 'W1', 'b1'], ['H'], domain=DOMAIN),
     make_node('AbsLinearRegression', ['H', 'W2', 'b2'], ['Y'], domain=DOMAIN)],
    'multi_fn', [X, W1, b1, W2, b2], [Y])

model_multi = make_model(graph, opset_imports=OPSETS,
                         functions=[relu_linear, abs_lr])
check_model(model_multi)

sess_m = ort.InferenceSession(
    model_multi.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.random.randn(3, 4).astype(np.float32)
w1 = np.random.randn(4, 8).astype(np.float32) * 0.1
b1_val = np.zeros(8, dtype=np.float32)
w2 = np.random.randn(8, 2).astype(np.float32) * 0.1
b2_val = np.zeros((1, 2), dtype=np.float32)

result = sess_m.run(None, {
    'X': x, 'W1': w1, 'b1': b1_val, 'W2': w2, 'b2': b2_val})[0]

h_np = np.maximum(0, x @ w1 + b1_val)
y_np = np.abs(h_np @ w2 + b2_val)

print(f'Two functions in one model:')
print(f'  Functions: {[f.name for f in model_multi.functions]}')
print(f'  Graph nodes: {[n.op_type for n in model_multi.graph.node]}')
print(f'  Output shape: {result.shape}')
print(f'  Match: {np.allclose(result, y_np, atol=1e-5)}')

<a id='section-6'></a>
## Section 6: Exercise 5 — Inspect Function Definitions

In [ ]:
def inspect_functions(model):
    """Print detailed info about all functions in a model."""
    print(f'Model has {len(model.functions)} function(s):')
    print('=' * 60)
    for fn in model.functions:
        print(f'\n  {fn.domain}::{fn.name}')
        print(f'    Inputs:  {list(fn.input)}')
        print(f'    Outputs: {list(fn.output)}')
        print(f'    Nodes:   {len(fn.node)}')
        for i, n in enumerate(fn.node):
            print(f'      [{i}] {n.op_type}: {list(n.input)} → {list(n.output)}')
        print(f'    Attributes: {list(fn.attribute) or "(none)"}')

inspect_functions(model_multi)

<a id='section-7'></a>
## Section 7: Visualization — Function Library

In [ ]:
functions_info = [
    ('AbsLinearRegression', ['MatMul', 'Add', 'Abs'], '|XA + B|'),
    ('ReluLinear', ['MatMul', 'Add', 'Relu'], 'Relu(XW + b)'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, ops, formula) in zip(axes, functions_info):
    ax.set_title(f'{DOMAIN}::{name}\n{formula}', fontsize=12, fontweight='bold')
    ax.axis('off')
    ax.set_xlim(-1, 5)
    ax.set_ylim(-0.5, len(ops) + 1)

    colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']
    for i, op in enumerate(ops):
        y = len(ops) - i
        ax.add_patch(plt.Rectangle((0.5, y-0.3), 3, 0.6,
                     facecolor=colors[i % len(colors)], alpha=0.3,
                     edgecolor=colors[i % len(colors)], linewidth=2))
        ax.text(2, y, op, ha='center', va='center', fontsize=11, fontweight='bold')
        if i > 0:
            ax.annotate('', xy=(2, y + 0.35), xytext=(2, y + 0.65),
                       arrowprops=dict(arrowstyle='->', lw=1.5, color='#555'))

plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: Challenge — SiLU Activation Function

### Task

Define the **SiLU** (Sigmoid Linear Unit) activation as an ONNX function:

$$\text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

This requires two ops: `Sigmoid` and `Mul`.

In [ ]:
silu_fn = make_function(
    DOMAIN, 'SiLU',
    ['X'], ['Y'],
    [make_node('Sigmoid', ['X'], ['sig_x']),
     make_node('Mul', ['X', 'sig_x'], ['Y'])],
    OPSETS, [])

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, None])

graph = make_graph(
    [make_node('SiLU', ['X'], ['Y'], domain=DOMAIN)],
    'silu_model', [X], [Y])
model_silu = make_model(graph, opset_imports=OPSETS, functions=[silu_fn])
check_model(model_silu)

sess_silu = ort.InferenceSession(
    model_silu.SerializeToString(), providers=['CPUExecutionProvider'])

x_range = np.linspace(-6, 6, 200).reshape(1, -1).astype(np.float32)
y_onnx = sess_silu.run(None, {'X': x_range})[0]

def silu_np(x):
    return x / (1 + np.exp(-x))

y_np = silu_np(x_range)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_range.flatten(), y_onnx.flatten(), 'b-', linewidth=2, label='ONNX SiLU')
ax.plot(x_range.flatten(), y_np.flatten(), 'r--', linewidth=2, label='NumPy SiLU')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('SiLU(x)', fontsize=12)
ax.set_title('SiLU Activation: x · σ(x) — Implemented as ONNX Function',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Match: {np.allclose(y_onnx, y_np, atol=1e-6)}')

---

## Summary

| Exercise | Skill |
|----------|-------|
| 1 | Create and use a basic function |
| 2 | Chain function calls |
| 3 | Build neural network layer as function |
| 4 | Multiple functions in one model |
| 5 | Inspect function definitions |
| 6 | Visualize function library |
| Challenge | Implement SiLU activation |

**Next:** [Parsing and Checker](../07_Parsing_and_Checker/) — Text format, validation, shape inference.